# P300 BCI Speller: Exploratory Data Analysis & Preprocessing Pipeline

This notebook fulfills the requirement to perform initial exploratory data analysis on the un-preprocessed EEG data.
We will:
1. Load a single subject from the `BNCI2014_009` dataset.
2. Plot the raw signal to observe noise and eye-blinks.
3. Show the channel layout / 10-20 montage.
4. Epoch the data and plot the Grand Average P300 Event-Related Potential (ERP) to confirm signal presence.


In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('../src'))

import mne
import numpy as np
import matplotlib.pyplot as plt
from moabb.datasets import BNCI2014_009

# Load Dataset
ds = BNCI2014_009()
data = ds.get_data(subjects=[1])[1]
session = list(data.keys())[0]
run = list(data[session].keys())[0]
raw = data[session][run]

raw.pick_types(eeg=True)
print(raw.info)


## 1. Raw EEG Signal

Plotting a 10-second window to observe the continuous EEG signal. Notice the low-frequency drifts and occasional sharp peaks (usually EOG/blink artifacts) before bandpass and ICA filtering.


In [ ]:
%matplotlib inline
fig = raw.plot(duration=10.0, scalings={'eeg': 50e-6}, title="Raw EEG (Subject 1)", show=False)
plt.show()


## 2. Channel Layout

The dataset uses a subset of the standard 10-20 system. Below is the sensor placement montage.


In [ ]:
# Set standard montage
montage = mne.channels.make_standard_montage('standard_1020')
raw.set_montage(montage)

fig = raw.plot_sensors(show_names=True, show=False)
plt.show()


## 3. The P300 Event-Related Potential (ERP)

To see the P300 component, we need to filter, epoch the data, and average over many trials to increase the Signal-to-Noise Ratio (SNR).


In [ ]:
# Apply standard filters
raw.filter(0.1, 30.0, verbose=False)
raw.notch_filter(freqs=50, verbose=False)
raw.set_eeg_reference('average', projection=False, verbose=False)

# Epoching (-200ms to 800ms)
events, event_id = mne.events_from_annotations(raw, verbose=False)
epochs = mne.Epochs(
    raw, events, 
    event_id={'Target': event_id['Target'], 'NonTarget': event_id['NonTarget']},
    tmin=-0.2, tmax=0.8, baseline=(-0.2, 0), preload=True, verbose=False
)

# Plotting the ERP for Target vs NonTarget on channel 'Cz'
target_evoked = epochs['Target'].average()
nontarget_evoked = epochs['NonTarget'].average()

fig, ax = plt.subplots(figsize=(10, 5))
mne.viz.plot_compare_evokeds(
    dict(Target=target_evoked, NonTarget=nontarget_evoked),
    picks=['Cz'], axes=ax, show=False, colors={'Target': 'blue', 'NonTarget': 'orange'}
)
plt.title("P300 Grand Average ERP on Channel Cz", fontsize=14)
plt.show()
